# Benchmark Plots — Interactive Plotly

Visualizations for the Python web framework benchmark.  
Generated by `benchmark_plots.py` — compatible with **Dash** for later deployment.

| Chart | Function | Data |
|---|---|---|
| Request box plot | `box_requests()` | rounds.csv |
| Energy box plot | `box_energy()` | rounds.csv |
| Energy line chart | `line_energy()` | rounds.csv |
| Full framework dashboard | `dashboard_framework()` | rounds.csv |


In [1]:
import sys
!{sys.executable} -m pip install plotly==6.7.0 pandas

In [2]:
from pathlib import Path
import pandas as pd
import plotly.io as pio
pio.renderers.default = "iframe_connected"
from benchmark_plots import (
    load_rounds,
    box_requests, box_energy, line_energy,
    dashboard_framework,
    ENDPOINTS, REQUEST_METRICS, ENERGY_METRICS,
    _parse_version,
)

# Adjust to the path of your rounds.csv
ROUNDS_CSV = Path("rounds_filtered.csv")

df = load_rounds(ROUNDS_CSV)
frameworks = sorted(df["framework"].unique())
print(f"Frameworks: {frameworks}")
print(f"\nVersions per framework:")
for fw in frameworks:
    versions = sorted(df[df['framework'] == fw]['framework_version'].unique(), key=_parse_version)
    print(f"  {fw}: {versions}")


Frameworks: ['aiohttp', 'baize', 'django', 'fastapi', 'muffin', 'starlette']

Versions per framework:
  aiohttp: ['3.7.0', '3.7.1', '3.7.2', '3.7.3', '3.7.4', '3.9.0', '3.9.1', '3.9.2', '3.9.3', '3.9.4', '3.9.5', '3.10.0', '3.10.1', '3.10.2', '3.10.3', '3.10.4', '3.10.5', '3.10.6', '3.10.7', '3.10.8', '3.10.9', '3.10.10', '3.10.11', '3.11.0', '3.11.1', '3.11.2', '3.11.3', '3.11.4', '3.11.5', '3.11.6', '3.11.7', '3.11.8', '3.11.9', '3.11.10', '3.11.11', '3.11.12', '3.11.13', '3.11.14', '3.11.15', '3.11.16', '3.11.17', '3.11.18', '3.12.0', '3.12.1', '3.12.2', '3.12.3', '3.12.4', '3.12.6', '3.12.7', '3.12.8', '3.12.9', '3.12.10', '3.12.11', '3.12.12', '3.12.13', '3.12.14', '3.12.15', '3.13.0', '3.13.1', '3.13.2', '3.13.3']
  baize: ['0.12.0', '0.12.1', '0.13.0', '0.14.0', '0.14.1', '0.15.0', '0.16.0', '0.16.1', '0.17.0', '0.17.1', '0.18.0', '0.18.1', '0.18.2', '0.19.0', '0.19.1', '0.20.0', '0.20.1', '0.20.2', '0.20.3', '0.20.4', '0.20.5', '0.20.6', '0.20.7', '0.20.8', '0.21.0', '0.22.0', 

## 1. Box Plots — Response Time per Endpoint


In [3]:
# Select the framework and endpoint
FRAMEWORK = frameworks[0]   # change as needed
ENDPOINT  = "api"           # "api" | "html" | "upload"
METRIC    = "rt_p95"        # see REQUEST_METRICS for options

fig = box_requests(df, framework=FRAMEWORK, endpoint=ENDPOINT, metric=METRIC)
fig.show()


In [4]:
# All endpoints at once — for one framework
import plotly.io as pio

for ep in ENDPOINTS:
    fig = box_requests(df, framework=FRAMEWORK, endpoint=ep, metric="rt_p95")
    fig.show()


## 2. Box Plots — Energy per Version


In [5]:
# Energy box plot — total energy (kWh)
print(df.columns.tolist())

fig = box_energy(df, framework=FRAMEWORK, metric="emission_energy_consumed")
fig.show()


['exc_n', 'framework', 'framework_version', 'emission_duration', 'emission_emissions', 'emission_cpu_energy', 'emission_gpu_energy', 'emission_ram_energy', 'emission_energy_consumed', 'emission_emissions_rate', 'emission_cpu_power', 'emission_ram_power', 'emission_power_mean', 'emission_power_median', 'emission_co2eq_year', 'python_version', 'cpu_model', 'cpu_count', 'ram_total_size', 'country_name', 'region', 'on_cloud', 'api_requests_total', 'api_requests_success', 'api_requests_error', 'api_success_rate', 'api_rt_mean', 'api_rt_std', 'api_rt_min', 'api_rt_max', 'api_rt_p50', 'api_rt_p75', 'api_rt_p90', 'api_rt_p95', 'api_rt_p99', 'api_throughput_rps', 'api_response_delay_mean', 'api_response_delay_p95', 'api_response_read_mean', 'api_response_read_p95', 'api_request_write_mean', 'api_request_write_p95', 'html_requests_total', 'html_requests_success', 'html_requests_error', 'html_success_rate', 'html_rt_mean', 'html_rt_std', 'html_rt_min', 'html_rt_max', 'html_rt_p50', 'html_rt_p75',

In [6]:
# Compare CPU vs RAM in side-by-side subplots
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig_cpu = box_energy(df, framework=FRAMEWORK, metric="emission_cpu_energy")
fig_ram = box_energy(df, framework=FRAMEWORK, metric="emission_ram_energy")

combined = make_subplots(rows=1, cols=2,
    subplot_titles=["CPU Energy (kWh)", "RAM Energy (kWh)"])

for trace in fig_cpu.data:
    trace.showlegend = False
    combined.add_trace(trace, row=1, col=1)
for trace in fig_ram.data:
    trace.showlegend = False
    combined.add_trace(trace, row=1, col=2)

combined.update_layout(
    title=f"{FRAMEWORK.title()} — CPU vs RAM Energy",
    plot_bgcolor="white", paper_bgcolor="white",
    height=450, showlegend=False,
)
combined.show()


## 3. Line Chart — Energy Consumption per Version


In [7]:
# Line chart with mean ± std band
fig = line_energy(df, framework="fastapi", metric="emission_energy_consumed", show_ci=True)
fig.show()


## 4. Full Dashboard — One Framework


In [8]:
# 2x3 dashboard: API/HTML/Upload box plots (p95) + energy box + energy line
fig = dashboard_framework(df, framework=FRAMEWORK, energy_metric="emission_energy_consumed")
fig.show()

# Export as standalone HTML
out = f"dashboard_{FRAMEWORK}.html"
fig.write_html(out, include_plotlyjs="cdn")
print(f"Saved: {out}")


Saved: dashboard_aiohttp.html


## 5. Export All Charts as HTML (for Dash / sharing)


In [9]:
from benchmark_plots import export_all

# Export all charts organised by framework (Option A)
# include_plotlyjs="cdn"    -> smaller files, requires internet to open
# include_plotlyjs="inline" -> larger files, works offline

export_all(df, output_dir="plots", include_plotlyjs="cdn")



[aiohttp]
  requests/api_rt_mean.html
  requests/api_rt_p95.html
  requests/api_throughput_rps.html
  requests/html_rt_mean.html
  requests/html_rt_p95.html
  requests/html_throughput_rps.html
  requests/upload_rt_mean.html
  requests/upload_rt_p95.html
  requests/upload_throughput_rps.html
  energy/line_emission_energy_consumed.html
  energy/line_emission_cpu_energy.html
  energy/line_emission_ram_energy.html
  energy/line_emission_emissions.html
  energy/line_emission_duration.html
  energy/line_emission_emissions_rate.html

[baize]
  requests/api_rt_mean.html
  requests/api_rt_p95.html
  requests/api_throughput_rps.html
  requests/html_rt_mean.html
  requests/html_rt_p95.html
  requests/html_throughput_rps.html
  requests/upload_rt_mean.html
  requests/upload_rt_p95.html
  requests/upload_throughput_rps.html
  energy/line_emission_energy_consumed.html
  energy/line_emission_cpu_energy.html
  energy/line_emission_ram_energy.html
  energy/line_emission_emissions.html
  energy/line_em

{'aiohttp': [PosixPath('plots/aiohttp/requests/api_rt_mean.html'),
  PosixPath('plots/aiohttp/requests/api_rt_p95.html'),
  PosixPath('plots/aiohttp/requests/api_throughput_rps.html'),
  PosixPath('plots/aiohttp/requests/html_rt_mean.html'),
  PosixPath('plots/aiohttp/requests/html_rt_p95.html'),
  PosixPath('plots/aiohttp/requests/html_throughput_rps.html'),
  PosixPath('plots/aiohttp/requests/upload_rt_mean.html'),
  PosixPath('plots/aiohttp/requests/upload_rt_p95.html'),
  PosixPath('plots/aiohttp/requests/upload_throughput_rps.html'),
  PosixPath('plots/aiohttp/energy/line_emission_energy_consumed.html'),
  PosixPath('plots/aiohttp/energy/line_emission_cpu_energy.html'),
  PosixPath('plots/aiohttp/energy/line_emission_ram_energy.html'),
  PosixPath('plots/aiohttp/energy/line_emission_emissions.html'),
  PosixPath('plots/aiohttp/energy/line_emission_duration.html'),
  PosixPath('plots/aiohttp/energy/line_emission_emissions_rate.html')],
 'baize': [PosixPath('plots/baize/requests/api

## 6. Dash Skeleton (starting point for deployment)

Run `python dash_app.py` to start the dashboard server.


In [13]:
```
dash_code = '''
"""
dash_app.py — Interactive dashboard for benchmark results.
Install: pip install dash plotly pandas
Run:     python dash_app.py
"""
import dash
from dash import dcc, html, Input, Output, callback
import pandas as pd

from benchmark_plots import (
    load_rounds, box_requests, box_energy, line_energy,
    ENDPOINTS, REQUEST_METRICS, ENERGY_METRICS,
)

df = load_rounds("rounds.csv")
frameworks = sorted(df["framework"].unique())

app = dash.Dash(__name__, title="Framework Benchmark")

app.layout = html.Div([
    html.H1("Python Web Framework Benchmark", style={"fontFamily": "Inter, sans-serif"}),

    # Row 1: selectors
    html.Div([
        html.Div([
            html.Label("Framework"),
            dcc.Dropdown(
                id="dd-framework",
                options=[{"label": f, "value": f} for f in frameworks],
                value=frameworks[0], clearable=False,
            ),
        ], style={"width": "20%", "display": "inline-block", "marginRight": "2%"}),

        html.Div([
            html.Label("Endpoint"),
            dcc.Dropdown(
                id="dd-endpoint",
                options=[{"label": e.upper(), "value": e} for e in ENDPOINTS],
                value="api", clearable=False,
            ),
        ], style={"width": "20%", "display": "inline-block", "marginRight": "2%"}),

        html.Div([
            html.Label("Request Metric"),
            dcc.Dropdown(
                id="dd-req-metric",
                options=[{"label": v, "value": k} for k, v in REQUEST_METRICS.items()],
                value="rt_p95", clearable=False,
            ),
        ], style={"width": "30%", "display": "inline-block", "marginRight": "2%"}),

        html.Div([
            html.Label("Energy Metric"),
            dcc.Dropdown(
                id="dd-energy-metric",
                options=[{"label": v, "value": k} for k, v in ENERGY_METRICS.items()],
                value="energy_consumed", clearable=False,
            ),
        ], style={"width": "22%", "display": "inline-block"}),
    ], style={"marginBottom": "20px"}),

    # Row 2: box plots
    html.Div([
        html.Div([dcc.Graph(id="graph-box-requests")],
                 style={"width": "49%", "display": "inline-block"}),
        html.Div([dcc.Graph(id="graph-box-energy")],
                 style={"width": "49%", "display": "inline-block", "marginLeft": "2%"}),
    ]),

    # Row 3: energy line chart
    dcc.Graph(id="graph-line-energy"),
], style={"maxWidth": "1400px", "margin": "0 auto", "padding": "20px"})


@callback(
    Output("graph-box-requests", "figure"),
    Input("dd-framework", "value"),
    Input("dd-endpoint", "value"),
    Input("dd-req-metric", "value"),
)
def update_box_requests(framework, endpoint, metric):
    return box_requests(df, framework=framework, endpoint=endpoint, metric=metric)


@callback(
    Output("graph-box-energy", "figure"),
    Input("dd-framework", "value"),
    Input("dd-energy-metric", "value"),
)
def update_box_energy(framework, metric):
    return box_energy(df, framework=framework, metric=metric)


@callback(
    Output("graph-line-energy", "figure"),
    Input("dd-framework", "value"),
    Input("dd-energy-metric", "value"),
)
def update_line_energy(framework, metric):
    return line_energy(df, framework=framework, metric=metric, show_ci=True)


if __name__ == "__main__":
    app.run(debug=True)
'''

with open("dash_app.py", "w") as f:
    f.write(dash_code)
print("dash_app.py generated.")
```


dash_app.py gerado.
